In [1]:
import json
import math
import warnings
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from transformers import logging as hf_logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
)
import pickle
from tqdm import tqdm

hf_logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*Token indices sequence length.*")
warnings.filterwarnings("ignore", message=".*Some weights of Roberta.*")

In [2]:
# Configuration

MODEL_NAME   = "roberta-large"

SPECIAL_ROLE_TOKENS = [
    "<SUB>", "</SUB>",
    "<OBJ>", "</OBJ>",
    "<VRB>", "</VRB>",
    "<MNR>", "</MNR>",
    "<LOC>", "</LOC>",
    "<TMP>", "</TMP>",
    "<CAU>", "</CAU>",
    "<NEG>", "</NEG>",
    "<MOD>", "</MOD>",
    "<ADV>", "</ADV>",
    "<PRP>", "</PRP>",
    "<OB2>", "</OB2>",
    "<BNF>", "</BNF>",
    "<EPT>", "</EPT>",
]

DATASET_PATH = "/kaggle/input/datasets/worldseeker/completecontrastivedataset/siamese_samples_with_srl.json"
BASE_DIR     = "/kaggle/working"
SEEDS        = [42 , 123, 7, 21, 99, 555]
SPLIT_SEED   = 42          # all seeds share the same test set

# Architecture 
MAX_LEN      = 128
HIDDEN_SIZE  = 1024        # roberta-large
PROJ_DIM     = 256
DROPOUT      = 0.3

# Training
BATCH_SIZE          = 16
GRAD_ACCUM_STEPS    = 2    # effective batch = 32
EPOCHS              = 15
PATIENCE            = 5    # early stop on val F1
LR_ENCODER          = 2e-5
LR_HEADS            = 1e-4  # heads 
WEIGHT_DECAY        = 1e-2
WARMUP_RATIO        = 0.1
FREEZE_EPOCHS       = 1    # freeze encoder for first epochs

# Loss
CONTRASTIVE_TEMP    = 0.07  
BCE_POS_WEIGHT      = 2.0   # counter 76/24 imbalance
LABEL_SMOOTHING     = 0.02  # prevent overconfident BCE
CONTRASTIVE_WEIGHT  = 0.5   # weight applied to contrastive loss only; BCE weight = 1.0

TEST_SIZE  = 0.15
VAL_SIZE   = 0.15
USE_SRL    = True       # False: use raw CVE/Technique text, skipping SRL markup


# Reproducibility Seed Setter

In [3]:
def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"[Seed] {seed}")

# Data Load

In [4]:

def load_srl_dataset(json_path: str) -> pd.DataFrame:
    try:
        with open(json_path) as f:
            data = json.load(f).get("samples", [])
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"[Error] {e}")
        return pd.DataFrame()
    samples = []
    for sample in tqdm(data, desc="[Data] Processing SRL"):
        try:
            cve_text  = sample.get("CVE_markup", "").strip() or sample.get("CVE_text", "")
            
            tech_text = sample.get("Technique_markup", "").strip() or sample.get("Technique_text", "")
            if not cve_text.strip() or not tech_text.strip():
                continue
            samples.append({
                "CVE_ID":         sample.get("CVE_ID", ""),
                "CVE_text":       cve_text,
                "Technique_text": tech_text,
                "label":          int(sample.get("label", 0)),
                "role_score":     0.0, # signal testing
            })
        except Exception as e:
            print(f"[Warning] Skipping sample: {e}")
    df = pd.DataFrame(samples)
    print(f"[Data] {len(df)} samples | "
          f"Pos={df['label'].sum()} Neg={(df['label']==0).sum()}")
    return df

def load_raw_dataset(json_path: str) -> pd.DataFrame:
    """Load dataset using raw CVE/Technique text without any SRL markup."""
    try:
        with open(json_path) as f:
            data = json.load(f).get("samples", [])
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"[Error] {e}")
        return pd.DataFrame()

    samples = []
    for sample in tqdm(data, desc="[Data] Processing RAW"):
        try:
            cve_text  = sample.get("CVE_text",       "")
            tech_text = sample.get("Technique_text", "")
            if not cve_text.strip() or not tech_text.strip():
                continue
            samples.append({
                "CVE_ID":         sample.get("CVE_ID", ""),
                "CVE_text":       cve_text,
                "Technique_text": tech_text,
                "label":          int(sample.get("label", 0)),
                "role_score":     0.0,
            })
        except Exception as e:
            print(f"[Warning] Skipping sample: {e}")

    df = pd.DataFrame(samples)
    print(f"[Data] {len(df)} samples | "
          f"Pos={df['label'].sum()} Neg={(df['label']==0).sum()}")
    return df


In [5]:

# Dataset

class SiameseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer,
                 max_len: int   = MAX_LEN,
                 role_min: float = None,
                 role_max: float = None):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

        r = df["role_score"].values.astype(np.float32)
        self.role_min = float(r.min()) if role_min is None else role_min
        self.role_max = float(r.max()) if role_max is None else role_max
        rng = max(self.role_max - self.role_min, 1e-8)
        self.role_weights  = (r - self.role_min) / rng
        self.binary_labels = df["label"].values.astype(np.float32)

    @property
    def norm_stats(self):
        return self.role_min, self.role_max

    def __len__(self):
        return len(self.df)

    def _encode_cve(self, text: str) -> dict:
        return self.tokenizer(
            text, max_length=self.max_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )

    def _encode_technique(self, text: str) -> dict:
        tokens = self.tokenizer(
            text, add_special_tokens=True,
            truncation=False, return_tensors="pt"
        )
        ids  = tokens["input_ids"][0]
        mask = tokens["attention_mask"][0]

        if ids.shape[0] <= self.max_len:
            pad_len = self.max_len - ids.shape[0]
            ids  = torch.cat([ids, torch.full((pad_len,), self.tokenizer.pad_token_id)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
        else:
            half = (self.max_len - 2) // 2
            ids  = torch.cat([ids[:half+1], ids[-(self.max_len - half - 1):]])
            mask = torch.ones(self.max_len, dtype=torch.long)

        return {"input_ids": ids.unsqueeze(0), "attention_mask": mask.unsqueeze(0)}

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        cve_enc  = self._encode_cve(row["CVE_text"])
        tech_enc = self._encode_technique(row["Technique_text"])
        
        return {
            "cve_input_ids":       cve_enc["input_ids"].squeeze(0),
            "cve_attention_mask":  cve_enc["attention_mask"].squeeze(0),
            "tech_input_ids":      tech_enc["input_ids"].squeeze(0),
            "tech_attention_mask": tech_enc["attention_mask"].squeeze(0),
            "binary_labels":       torch.tensor(self.binary_labels[idx], dtype=torch.float),
            "role_weights":        torch.tensor(self.role_weights[idx],  dtype=torch.float),
            "CVE_text":            row["CVE_text"],
            "Technique_text":      row["Technique_text"],
        }



# Model

In [6]:
# Model

class SoftAlignAttention(nn.Module):
    """ESIM-style cross-sentence soft alignment."""
    def forward(self, a, b, mask_a, mask_b):
        sim      = torch.bmm(a, b.transpose(1, 2))
        mask_b_e = (mask_b == 0).unsqueeze(1).expand_as(sim)
        mask_a_e = (mask_a == 0).unsqueeze(2).expand_as(sim)
        attn_a   = F.softmax(sim.masked_fill(mask_b_e, -1e4), dim=2)
        attn_b   = F.softmax(sim.masked_fill(mask_a_e, -1e4).transpose(1, 2), dim=2)
        return torch.bmm(attn_a, b), torch.bmm(attn_b, a)


class AttentionPooling(nn.Module):
    """Weighted mean pooling using a learned attention scalar per token."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, token_emb, attention_mask):
        scores  = self.attn(token_emb).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e4)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        return (token_emb * weights).sum(dim=1)


class SemanticLinkModel(nn.Module):
    """SemanticLink architecture"""

    def __init__(self,
                 model_name:  str   = MODEL_NAME,
                 hidden_size: int   = HIDDEN_SIZE,
                 proj_dim:    int   = PROJ_DIM,
                 dropout:     float = DROPOUT):
        super().__init__()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.encoder = AutoModel.from_pretrained(model_name)

        self.align = SoftAlignAttention()
        self.pool  = AttentionPooling(hidden_size)

        self.proj = nn.Sequential(
            nn.Linear(hidden_size, proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        self.contrast_head = nn.Sequential(
            nn.Linear(proj_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, proj_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 4, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )

    # Encoder freeze / unfreeze 
    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True

    # Forward 
    def _encode(self, input_dict):
        return self.encoder(**input_dict).last_hidden_state

    def forward(self, cve_input, tech_input, binary_labels=None):
        cve_seq  = self._encode(cve_input)
        tech_seq = self._encode(tech_input)
        cve_mask  = cve_input["attention_mask"]
        tech_mask = tech_input["attention_mask"]
        # [ABLATION] Cross-attention removed: pool raw encoder output directly
        # aligned_cve, aligned_tech = self.align(cve_seq, tech_seq, cve_mask, tech_mask)
        # cve_combined  = (cve_seq  + aligned_cve)  / 2.0
        # tech_combined = (tech_seq + aligned_tech) / 2.0

        cve_emb  = self.proj(self.pool(cve_seq,  cve_mask))  
        tech_emb = self.proj(self.pool(tech_seq, tech_mask))

        diff     = torch.abs(cve_emb - tech_emb)
        prod     = cve_emb * tech_emb
        features = torch.cat([cve_emb, tech_emb, diff, prod], dim=1)
        logit    = self.classifier(features).squeeze(-1)

        cont_loss = torch.tensor(0.0, device=logit.device)
        if binary_labels is not None:
            c_cve  = F.normalize(self.contrast_head(cve_emb),  dim=-1)
            c_tech = F.normalize(self.contrast_head(tech_emb), dim=-1)
            cont_loss = supervised_contrastive_loss(
                torch.stack([c_cve, c_tech], dim=1),
                binary_labels, temperature=CONTRASTIVE_TEMP
            )
        return logit, cve_emb, tech_emb, cont_loss


    def get_optimizer_groups(self, lr_encoder: float, lr_heads: float,
                              weight_decay: float):
        n_layers   = self.encoder.config.num_hidden_layers
        decay_rate = 0.9

        encoder_params = []
        for layer_idx in range(n_layers):
            layer_lr = lr_encoder * (decay_rate ** (n_layers - layer_idx))
            params   = [p for n, p in self.encoder.named_parameters()
                        if f"layer.{layer_idx}." in n and p.requires_grad]
            if params:
                encoder_params.append({"params": params, "lr": layer_lr,
                                        "weight_decay": weight_decay})

        embedding_params = [p for n, p in self.encoder.named_parameters()
                            if "embeddings" in n or "pooler" in n]
        if embedding_params:
            encoder_params.append({
                "params": embedding_params,
                "lr": lr_encoder * (decay_rate ** n_layers),
                "weight_decay": weight_decay
            })

        head_params = (list(self.proj.parameters()) +
                       list(self.contrast_head.parameters()) +
                       list(self.classifier.parameters()) +
                       list(self.pool.parameters()))
        head_group  = {"params": head_params, "lr": lr_heads,
                       "weight_decay": weight_decay}

        return encoder_params + [head_group]



In [7]:


# Loss Functions

def supervised_contrastive_loss(features: torch.Tensor,
                                 labels:   torch.Tensor,
                                 temperature: float = 0.05) -> torch.Tensor:
    """SupCon loss with in-batch negatives (Khosla et al., 2020)."""
    B    = features.size(0)
    flat = features.view(2 * B, -1)
    lab  = labels.repeat(2).long()

    sim       = torch.mm(flat, flat.T) / temperature
    self_mask = torch.eye(2 * B, dtype=torch.bool, device=features.device)
    sim.masked_fill_(self_mask, -1e4)

    pos_mask  = (lab.unsqueeze(0) == lab.unsqueeze(1)) & ~self_mask
    log_prob  = F.log_softmax(sim, dim=1)
    pos_count = pos_mask.sum(dim=1).clamp(min=1)
    loss      = -(log_prob * pos_mask.float()).sum(dim=1) / pos_count
    return loss.mean()


class SmoothedBCELoss(nn.Module):
    """BCE with label smoothing + class imbalance weighting + role weighting."""
    def __init__(self, pos_weight: float = BCE_POS_WEIGHT,
                 smoothing: float = LABEL_SMOOTHING):
        super().__init__()
        self.pos_weight = pos_weight
        self.smoothing  = smoothing

    def forward(self, logits: torch.Tensor,
                labels:  torch.Tensor,
                weights: torch.Tensor) -> torch.Tensor:
        smooth_labels = labels * (1 - self.smoothing) + 0.5 * self.smoothing
        pos_w = torch.tensor(self.pos_weight, device=logits.device)
        bce   = F.binary_cross_entropy_with_logits(
            logits, smooth_labels, pos_weight=pos_w, reduction="none"
        )
        return bce.mean()


In [8]:

# Utilities

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def platt_calibrate(val_logits: np.ndarray, val_labels: np.ndarray):
    cal = LogisticRegression(C=1e6, max_iter=1000)
    cal.fit(val_logits.reshape(-1, 1), val_labels.astype(int))
    return cal


In [9]:


# Data Splitting

def stratified_split(df: pd.DataFrame, random_state: int = SPLIT_SEED):
    if "CVE_ID" not in df.columns:
        raise ValueError("CVE_ID column required for leakage-free splitting")

    # Split CVE IDs into train+val vs test

    gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE,
                                  random_state=random_state)
    train_val_idx, test_idx = next(
        gss_test.split(df, groups=df["CVE_ID"])
    )
    train_val_df = df.iloc[train_val_idx]
    test_df      = df.iloc[test_idx]

    # Split train+val CVE IDs into train vs val
    rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=rel_val,
                                 random_state=random_state)
    train_idx, val_idx = next(
        gss_val.split(train_val_df, groups=train_val_df["CVE_ID"])
    )
    train_df = train_val_df.iloc[train_idx]
    val_df   = train_val_df.iloc[val_idx]

    # Verify no CVE leakage
    train_cves = set(train_df["CVE_ID"])
    val_cves   = set(val_df["CVE_ID"])
    test_cves  = set(test_df["CVE_ID"])
    assert len(train_cves & test_cves) == 0, "Train/Test CVE leak!"
    assert len(train_cves & val_cves)  == 0, "Train/Val CVE leak!"
    assert len(val_cves   & test_cves) == 0, "Val/Test CVE leak!"

    print(f"[Split] Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}")
    print(f"  CVEs  Train={train_df['CVE_ID'].nunique()}  "
          f"Val={val_df['CVE_ID'].nunique()}  "
          f"Test={test_df['CVE_ID'].nunique()}")
    return train_df, val_df, test_df


def build_loaders(train_df, val_df, test_df, tokenizer, seed: int = SPLIT_SEED):
    train_ds = SiameseDataset(train_df, tokenizer)
    rmin, rmax = train_ds.norm_stats
    val_ds   = SiameseDataset(val_df,  tokenizer, role_min=rmin, role_max=rmax)
    test_ds  = SiameseDataset(test_df, tokenizer, role_min=rmin, role_max=rmax)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True, generator=g)
    val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    return train_loader, val_loader, test_loader, rmin, rmax


In [10]:
# Training

def run_epoch(model, loader, criterion, optimizer, scheduler,
              scaler, device, training: bool,
              grad_accum: int = GRAD_ACCUM_STEPS) -> tuple:
    model.train() if training else model.eval()
    total_loss = n_batches = 0
    all_logits, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for step, batch in enumerate(tqdm(loader,
                desc="train" if training else "eval ", leave=False)):
            cve_in  = {"input_ids": batch["cve_input_ids"].to(device),
                       "attention_mask": batch["cve_attention_mask"].to(device)}
            tech_in = {"input_ids": batch["tech_input_ids"].to(device),
                       "attention_mask": batch["tech_attention_mask"].to(device)}
            b_labs  = batch["binary_labels"].to(device)
            weights = batch["role_weights"].to(device)

            with autocast('cuda', enabled=(scaler is not None)):
                logits, _, _, cont_loss = model(cve_in, tech_in, b_labs)
                bce_loss = criterion(logits, b_labs, weights)
                loss = bce_loss + CONTRASTIVE_WEIGHT * cont_loss
                loss = loss / grad_accum

            if training:
                if scaler is not None:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

                if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scale_before = scaler.get_scale()    
                        scaler.step(optimizer)
                        scaler.update()
                        scale_after = scaler.get_scale()

                        if scheduler is not None and scale_after >= scale_before:
                            scheduler.step()
                    
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        if scheduler is not None:
                            scheduler.step()

                    optimizer.zero_grad()

            total_loss += loss.item() * grad_accum
            n_batches  += 1
            all_logits.extend(logits.detach().cpu().float().numpy())
            all_labels.extend(b_labs.cpu().numpy())

    logits_arr = np.array(all_logits)
    labels_arr = np.array(all_labels)
    probs      = sigmoid_np(logits_arr)
    preds      = (probs >= 0.5).astype(int)

    acc   = (preds == labels_arr.astype(int)).mean()
    ep_f1 = f1_score(labels_arr, preds, zero_division=0)
    return total_loss / n_batches, float(acc), float(ep_f1), logits_arr, labels_arr

def train_model(train_df, val_df, results_dir, tokenizer, device):
    train_loader, val_loader, _, rmin, rmax = build_loaders(
        train_df, val_df, val_df, tokenizer)

    model     = SemanticLinkModel().to(device)
    model.encoder.resize_token_embeddings(len(tokenizer))

    if USE_SRL:
    # Seed new role-tag embeddings from related words
        with torch.no_grad():
            emb = model.encoder.embeddings.word_embeddings.weight
            seed_words = {
                "<SUB>": "subject",  "</SUB>": "subject",
                "<OBJ>": "object",   "</OBJ>": "object",
                "<VRB>": "action",   "</VRB>": "action",
                "<MNR>": "manner",   "</MNR>": "manner",
                "<LOC>": "location", "</LOC>": "location",
                "<TMP>": "time",     "</TMP>": "time",
                "<CAU>": "cause",    "</CAU>": "cause",
                "<NEG>": "not",      "</NEG>": "not",
                "<MOD>": "modal",    "</MOD>": "modal",
                "<ADV>": "however",  "</ADV>": "however",
                "<PRP>": "purpose",  "</PRP>": "purpose",
                "<OB2>": "indirect", "</OB2>": "indirect",
                "<BNF>": "benefit",  "</BNF>": "benefit",
                "<EPT>": "endpoint", "</EPT>": "endpoint",
            }
            for tag, seed_word in seed_words.items():   # ← inside with block
                tag_id  = tokenizer.convert_tokens_to_ids(tag)
                seed_id = tokenizer.convert_tokens_to_ids(seed_word)
                if seed_id != tokenizer.unk_token_id:
                    emb[tag_id] = emb[seed_id].clone()
        print(f"[Model] Embedding matrix resized to {len(tokenizer)}")
    print(f"[Model] Embedding matrix size: {len(tokenizer)}")
    print(f"[Train] Freezing encoder for first {FREEZE_EPOCHS} epochs...")
    model.freeze_encoder()
    criterion = SmoothedBCELoss()

    # train heads only (frozen)
    head_params = (
        list(model.proj.parameters()) +
        list(model.contrast_head.parameters()) +
        list(model.classifier.parameters()) +
        list(model.pool.parameters())
    )
    optimizer = optim.AdamW(head_params, lr=LR_HEADS, weight_decay=WEIGHT_DECAY)

    # only the frozen phase steps
    freeze_steps  = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * FREEZE_EPOCHS
    warmup_steps  = int(freeze_steps * WARMUP_RATIO)
    scheduler     = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=freeze_steps)

    scaler = GradScaler('cuda') if torch.cuda.is_available() else None

    history = {k: [] for k in
               ("train_loss", "train_acc", "train_f1", "val_loss", "val_acc", "val_f1")}
    best_val_f1   = 0.0
    patience_ctr  = 0
    epochs_done   = 0
    best_path     = os.path.join(results_dir, "best_model.pth")

    for epoch in range(1, EPOCHS + 1):

        if epoch == FREEZE_EPOCHS + 1:
            print("[Train] Unfreezing encoder — full fine-tuning begins.")
            model.unfreeze_encoder()

            param_groups = model.get_optimizer_groups(
                LR_ENCODER, LR_HEADS, WEIGHT_DECAY)
            optimizer = optim.AdamW(param_groups)

            # only the remaining fine-tuning steps
            remaining_steps  = (
                math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
                * (EPOCHS - FREEZE_EPOCHS)
            )
            warmup_steps_ft = max(1, int(remaining_steps * WARMUP_RATIO))
            scheduler        = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps_ft,
                num_training_steps=remaining_steps,
            )
            print(f"[Train] Fresh optimizer+scheduler | "
                  f"remaining_steps={remaining_steps} "
                  f"warmup={warmup_steps_ft}")

        t_loss, t_acc, t_f1, _, _ = run_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            scaler, device, training=True)
        v_loss, v_acc, v_f1, v_logits, v_labels = run_epoch(
            model, val_loader, criterion, None, None,
            None, device, training=False)

        history["train_loss"].append(t_loss); history["train_acc"].append(t_acc)
        history["train_f1"].append(t_f1)
        history["val_loss"].append(v_loss);   history["val_acc"].append(v_acc)
        history["val_f1"].append(v_f1)

        print(f"Ep {epoch:02d}/{EPOCHS} | "
              f"Train loss={t_loss:.4f} acc={t_acc:.4f} F1={t_f1:.4f} | "
              f"Val   loss={v_loss:.4f} acc={v_acc:.4f} F1={v_f1:.4f}")

        epochs_done = epoch
        if v_f1 > best_val_f1:
            best_val_f1  = v_f1
            patience_ctr = 0
            torch.save(model.state_dict(), best_path)
            with open(os.path.join(results_dir, "best_val_metrics.json"), "w") as f:
                json.dump({"val_f1": v_f1, "val_acc": v_acc,
                           "val_loss": v_loss, "epoch": epoch}, f, indent=2)
            print(f"  ✓ Best saved  val_F1={v_f1:.4f}  @ epoch {epoch}")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stop @ epoch {epoch}  (best epoch {epoch - PATIENCE})")
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"[Train] Done. Epochs={epochs_done}  Best val_F1={best_val_f1:.4f}")
    plot_history(history, results_dir)

    _, _, _, v_logits_final, v_labels_final = run_epoch(
        model, val_loader, criterion, None, None, None, device, training=False)
    cal_model = platt_calibrate(v_logits_final, v_labels_final)

    with open(os.path.join(results_dir, "platt_cal.pkl"), "wb") as f:
        pickle.dump(cal_model, f)
    print(f"[Calibration] Platt model fitted on val set. Threshold fixed at 0.5")

    return model, epochs_done, cal_model, rmin, rmax  




In [11]:


# Evaluation

def evaluate_model(model, test_df, tokenizer, device, results_dir,
                   role_min, role_max, cal_model):
    test_ds  = SiameseDataset(test_df, tokenizer,
                               role_min=role_min, role_max=role_max)
    test_ldr = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
    criterion = SmoothedBCELoss()

    _, _, _, logits_arr, b_true = run_epoch(
        model, test_ldr, criterion, None, None, None, device, training=False)

    probs_arr = cal_model.predict_proba(logits_arr.reshape(-1, 1))[:, 1]
    b_preds   = (probs_arr >= 0.5).astype(int)

    report = classification_report(b_true.astype(int), b_preds, output_dict=True)
    try:
        auc = roc_auc_score(b_true, probs_arr)
        ap  = average_precision_score(b_true, probs_arr)
    except Exception:
        auc = ap = float("nan")

    pos_key = "1" if "1" in report else "1.0"
    metrics = {
        "accuracy":      float(report["accuracy"]),
        "precision":     float(report.get(pos_key, {}).get("precision", 0.0)),
        "recall":        float(report.get(pos_key, {}).get("recall",    0.0)),
        "f1_score":      float(report.get(pos_key, {}).get("f1-score",  0.0)),
        "roc_auc":       float(auc),
        "avg_precision": float(ap),
    }
    print("\n[Test Metrics]")
    for k, v in metrics.items():
        print(f"  {k:<16}: {v:.4f}")

    pd.DataFrame({
        "CVE_text":        [r["CVE_text"]       for r in [test_ds[i] for i in range(len(test_ds))]],
        "Technique_text":  [r["Technique_text"] for r in [test_ds[i] for i in range(len(test_ds))]],
        "true_label":      b_true.astype(int),
        "predicted_label": b_preds,
        "confidence_prob": probs_arr,
        "correct":         b_preds == b_true.astype(int),
    }).to_csv(os.path.join(results_dir, "test_results_detailed.csv"), index=False)

    cm = confusion_matrix(b_true.astype(int), b_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
    plt.title("Confusion Matrix"); plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "confusion_matrix.png")); plt.close()

    plt.figure(figsize=(7, 3))
    plt.hist(probs_arr[b_preds == b_true.astype(int)], alpha=0.6,
             bins=30, label="Correct",   color="#2980b9")
    plt.hist(probs_arr[b_preds != b_true.astype(int)], alpha=0.6,
             bins=30, label="Incorrect", color="#e74c3c")
    plt.xlabel("Confidence (sigmoid prob)"); plt.ylabel("Count")
    plt.legend(); plt.title("Confidence Distribution"); plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "confidence_distribution.png")); plt.close()

    if not math.isnan(auc):
        fpr, tpr, _ = roc_curve(b_true.astype(int), probs_arr)
        plt.figure(figsize=(5, 4))
        plt.plot(fpr, tpr, color="#2980b9", lw=2, label=f"AUC={auc:.4f}")
        plt.plot([0, 1], [0, 1], "k--", lw=1)
        plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title("ROC Curve"); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(results_dir, "roc_curve.png")); plt.close()

    if not math.isnan(ap):
        prec, rec, _ = precision_recall_curve(b_true.astype(int), probs_arr)
        plt.figure(figsize=(5, 4))
        plt.plot(rec, prec, color="#e67e22", lw=2, label=f"AP={ap:.4f}")
        plt.xlabel("Recall"); plt.ylabel("Precision")
        plt.title("Precision-Recall Curve"); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(results_dir, "pr_curve.png")); plt.close()


    return metrics

In [12]:
# Plots

def plot_history(history: dict, results_dir: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(axes,
                               ["loss", "acc", "f1"],
                               ["Loss", "Accuracy", "F1 (early stop)"]):
        ax.plot(history[f"train_{key}"], label="Train")
        ax.plot(history[f"val_{key}"],   label="Val")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(results_dir, "training_history.png")); plt.close()

# Main

In [13]:
# Main

def main():
    print("=" * 62)
    print("  SemanticLink — CVE-to-ATT&CK Siamese Model")
    print(f"  SRL mode: {'ON' if USE_SRL else 'OFF'}")
    print("=" * 62)

    df = load_srl_dataset(DATASET_PATH) if USE_SRL else load_raw_dataset(DATASET_PATH)
    if df.empty:
        print("[Error] Empty dataset."); return

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if USE_SRL:
        tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_ROLE_TOKENS})
        print(f"[Tokenizer] Vocabulary size: {len(tokenizer)}")  # 50293
    
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}  |  AMP={'yes' if torch.cuda.is_available() else 'no'}")

    os.makedirs(BASE_DIR, exist_ok=True)
    all_results = []

    print(f"\n[Split] SPLIT_SEED={SPLIT_SEED} — shared across all seeds")
    train_df, val_df, test_df = stratified_split(df, random_state=SPLIT_SEED)
    print(f"[Split] Test set locked: {len(test_df)} samples")

    for seed in SEEDS:
        print("\n" + "=" * 62)
        print(f"  SEED {seed}")
        print("=" * 62)
        set_all_seeds(seed)

        results_dir = os.path.join(BASE_DIR, f"results_seed_{seed}")
        os.makedirs(results_dir, exist_ok=True)

        best_path = os.path.join(results_dir, "best_model.pth")
        if os.path.exists(best_path):
            print(f"[Model] Loading {best_path}")
            model = SemanticLinkModel().to(device)

            model.encoder.resize_token_embeddings(len(tokenizer))
            
            model.load_state_dict(torch.load(best_path, map_location=device))
            _, _, _, rmin, rmax = build_loaders(train_df, val_df, test_df, tokenizer)
            epochs_done = 0
            cal_model = None
        else:
            model, epochs_done, cal_model, rmin, rmax = train_model(
                train_df, val_df, results_dir, tokenizer, device)

        if cal_model is None:
            # Platt calibration
            val_ds  = SiameseDataset(val_df, tokenizer, role_min=rmin, role_max=rmax)
            val_ldr = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            crit    = SmoothedBCELoss()
            _, _, _, v_log, v_lab = run_epoch(
                model, val_ldr, crit, None, None, None, device, training=False)
            cal_model = platt_calibrate(v_log, v_lab)
            with open(os.path.join(results_dir, "platt_cal.pkl"), "wb") as f:
                pickle.dump(cal_model, f)
            print("[Calibration] Platt fitted on val. Threshold fixed at 0.5")

        metrics = evaluate_model(model, test_df, tokenizer, device,
                          results_dir, rmin, rmax, cal_model)


        seed_result = {"model": MODEL_NAME, "seed": seed,
                       "srl_mode": USE_SRL,
                       "epochs_trained": epochs_done,
                       "threshold_used": 0.5,
                       **metrics}
        all_results.append(seed_result)
        with open(os.path.join(results_dir,
                               f"metrics_seed_{seed}.json"), "w") as f:
            json.dump(seed_result, f, indent=2)
        print(f"[Done] Seed {seed} → {results_dir}/")

    print("\n" + "=" * 62)
    print("  FINAL RESULTS  (mean ± std across seeds)")
    print("=" * 62)
    metric_keys = ["accuracy", "precision", "recall",
                   "f1_score", "roc_auc", "avg_precision"]
    summary = {"model": MODEL_NAME, "n_seeds": len(SEEDS), "srl_mode": USE_SRL}
    rows    = []
    for m in metric_keys:
        vals = [r[m] for r in all_results
                if not math.isnan(r.get(m, float("nan")))]
        mean_, std_ = float(np.mean(vals)), float(np.std(vals))
        summary[f"{m}_mean"] = mean_
        summary[f"{m}_std"]  = std_
        print(f"  {m:<18}: {mean_:.4f} ± {std_:.4f}")
        rows.append({"metric": m, "mean": mean_, "std": std_})

    with open(os.path.join(BASE_DIR, "final_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    pd.DataFrame(rows).to_csv(
        os.path.join(BASE_DIR, "final_summary.csv"), index=False)

    print(f"\n[Summary] Saved → {BASE_DIR}/final_summary.json + .csv")


if __name__ == "__main__":
    main()

  SemanticLink — CVE-to-ATT&CK Siamese Model
  SRL mode: ON


[Data] Processing SRL: 100%|██████████| 6602/6602 [00:00<00:00, 692010.37it/s]


[Data] 6602 samples | Pos=1647 Neg=4955


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Ep 13/15 | Train loss=2.0133 acc=0.9302 F1=0.8713 | Val   loss=2.0767 acc=0.9155 F1=0.8440


Ep 14/15 | Train loss=2.0055 acc=0.9347 F1=0.8789 | Val   loss=2.0735 acc=0.9135 F1=0.8421


Ep 15/15 | Train loss=2.0015 acc=0.9323 F1=0.8762 | Val   loss=2.0738 acc=0.9145 F1=0.8436
[Train] Done. Epochs=15  Best val_F1=0.8450


[Calibration] Platt model fitted on val set. Threshold fixed at 0.5



[Test Metrics]
  accuracy        : 0.9289
  precision       : 0.8519
  recall          : 0.8661
  f1_score        : 0.8589
  roc_auc         : 0.9646
  avg_precision   : 0.8723
[Done] Seed 555 → /kaggle/working/results_seed_555/

  FINAL RESULTS  (mean ± std across seeds)
  accuracy          : 0.9323 ± 0.0034
  precision         : 0.8520 ± 0.0093
  recall            : 0.8821 ± 0.0163
  f1_score          : 0.8667 ± 0.0073
  roc_auc           : 0.9669 ± 0.0016
  avg_precision     : 0.8905 ± 0.0100

[Summary] Saved → /kaggle/working/final_summary.json + .csv
